# Open Model Arena

Ask once. Compare four configurable open-weight models side by side.

In [1]:
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from queue import Queue
from threading import Barrier

import mercury as mr
from dotenv import load_dotenv
from openai import OpenAI

_ = load_dotenv(Path.cwd() / ".env")

In [2]:
MODEL_CATALOG = {
    "Qwen 3.8 2.4T": {"id": "qwen/qwen3.8-2.4t-a95b", "emoji": "🐉"},
    "Kimi K3": {"id": "moonshotai/kimi-k3", "emoji": "🌙"},
    "GLM 5.2": {"id": "z-ai/glm-5.2", "emoji": "🧠"},
    "DeepSeek V4 Pro": {"id": "deepseek/deepseek-v4-pro", "emoji": "🔍"},
    "MiniMax M3": {"id": "minimax/minimax-m3", "emoji": "🧩"},
    "Nemotron 3 Ultra": {"id": "nvidia/nemotron-3-ultra-550b-a55b", "emoji": "🟢"},
    "Mistral Small 4": {"id": "mistralai/mistral-small-2603", "emoji": "🌪️"},
    "GPT-OSS 120B": {"id": "openai/gpt-oss-120b", "emoji": "🛠️"},
    "Step 3.7 Flash": {"id": "stepfun/step-3.7-flash", "emoji": "⚡"},
}
PANEL_COUNT = 4
MAX_TOKENS = 1200

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise RuntimeError("OPENROUTER_API_KEY is missing. Add it to .env and restart Mercury.")

# Mercury reruns only cells below the widget that changed, so these are initialized once.
_arena_histories = [[] for _ in range(PANEL_COUNT)]
_arena_model_ids = [None for _ in range(PANEL_COUNT)]

def format_assistant_message(reasoning, answer, waiting=False):
    parts = []
    if reasoning:
        parts.extend(["🧠 **Reasoning**", reasoning, "---"])
    elif waiting:
        parts.extend(["🧠 **Reasoning**", "⏳ Waiting for the model...", "---"])
    parts.append("💬 **Answer**")
    parts.append(answer or ("⏳ Waiting for response..." if waiting else ""))
    return "\n\n".join(parts)

def stream_model(panel_index, model_id, messages, output_queue, request_barrier):
    """Stream one model in a worker and queue UI events for the main thread."""
    try:
        client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key,
            timeout=120.0,
            default_headers={
                "HTTP-Referer": "https://runmercury.com",
                "X-Title": "Mercury Open Model Arena",
            },
        )
        request_barrier.wait(timeout=30)
        output_queue.put(("started", panel_index, None))
        stream = client.chat.completions.create(
            model=model_id,
            messages=messages,
            max_tokens=MAX_TOKENS,
            stream=True,
            extra_body={"reasoning": {"exclude": False}},
        )
        for chunk in stream:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            reasoning_details = getattr(delta, "reasoning_details", None) or []
            reasoning_chunks = []
            for detail in reasoning_details:
                if hasattr(detail, "model_dump"):
                    detail = detail.model_dump()
                if isinstance(detail, dict):
                    reasoning_text = detail.get("text") or detail.get("summary")
                    if isinstance(reasoning_text, list):
                        reasoning_text = "".join(
                            part.get("text", "") if isinstance(part, dict) else str(part)
                            for part in reasoning_text
                        )
                    if reasoning_text:
                        reasoning_chunks.append(str(reasoning_text))
            if not reasoning_chunks:
                legacy_reasoning = getattr(delta, "reasoning", None)
                if legacy_reasoning:
                    reasoning_chunks.append(str(legacy_reasoning))
            for reasoning_text in reasoning_chunks:
                output_queue.put(("reasoning", panel_index, reasoning_text))
            token = getattr(delta, "content", None)
            if token:
                output_queue.put(("token", panel_index, token))
    except Exception as exc:
        output_queue.put(("error", panel_index, str(exc)))
    finally:
        output_queue.put(("done", panel_index, None))

In [3]:
model_names = list(MODEL_CATALOG)
model_choices = [f"{MODEL_CATALOG[name]['emoji']} {name}" for name in model_names]
choice_to_model = {
    f"{MODEL_CATALOG[name]['emoji']} {name}": name for name in model_names
}
default_models = ["Qwen 3.8 2.4T", "Kimi K3", "GLM 5.2", "DeepSeek V4 Pro"]
default_choices = [f"{MODEL_CATALOG[name]['emoji']} {name}" for name in default_models]
model_selectors = [
    mr.Select(
        label=f"Chat {panel_index + 1} model",
        value=default_choices[panel_index],
        choices=model_choices,
        position="sidebar",
        key=f"model-selector-{panel_index}",
    )
    for panel_index in range(PANEL_COUNT)
]
clear_button = mr.Button(
    label="Clear all chats",
    variant="outline",
    size="sm",
    position="sidebar",
    key="clear-arena",
)

In [4]:
active_model_names = [choice_to_model[selector.value] for selector in model_selectors]
active_models = [MODEL_CATALOG[name] for name in active_model_names]

if clear_button.value:
    for history in _arena_histories:
        history.clear()
    clear_button.value = False

for panel_index, model in enumerate(active_models):
    if _arena_model_ids[panel_index] != model["id"]:
        _arena_histories[panel_index].clear()
        _arena_model_ids[panel_index] = model["id"]

columns = mr.Columns(
    n=4,
    min_width="260px",
    gap="12px",
    border="1px solid #e5e7eb",
    key="arena-columns",
)
chats = []
for panel_index, (column, model, model_name) in enumerate(
    zip(columns, active_models, active_model_names)
):
    with column:
        chat = mr.Chat(
            height="700px",
            placeholder=f"Ask {model_name} something...",
            scroll_debounce=0.08,
        )
        for message in _arena_histories[panel_index]:
            if message["role"] == "user":
                chat.add(mr.Message(markdown=message["content"], role="user", emoji="👤"))
            else:
                chat.add(
                    mr.Message(
                        markdown=format_assistant_message(
                            message.get("reasoning", ""), message["content"]
                        ),
                        role="assistant",
                        emoji=model["emoji"],
                    )
                )
        chats.append(chat)

ColumnsBox(children=(ColumnOutput(layout=Layout(border_bottom='1px solid #e5e7eb', border_left='1px solid #e5e…

In [5]:
prompt = mr.ChatInput(
    placeholder="Ask all 4 models...",
    button_icon="➤",
    position="bottom",
    key="arena-prompt",
)

In [6]:
if prompt.value and prompt.value.strip():
    user_text = prompt.value.strip()
    response_messages = {}
    request_histories = {}

    for panel_index, model in enumerate(active_models):
        history = _arena_histories[panel_index]
        history.append({"role": "user", "content": user_text})
        request_histories[panel_index] = [
            {"role": message["role"], "content": message["content"]}
            for message in history
        ]
        chats[panel_index].add(mr.Message(markdown=user_text, role="user", emoji="👤"))
        response_message = mr.Message(
            markdown=format_assistant_message("", "", waiting=True),
            role="assistant",
            emoji=model["emoji"],
        )
        chats[panel_index].add(response_message)
        response_messages[panel_index] = response_message

    events = Queue()
    request_barrier = Barrier(PANEL_COUNT)
    answers = {panel_index: "" for panel_index in range(PANEL_COUNT)}
    reasoning_outputs = {panel_index: "" for panel_index in range(PANEL_COUNT)}
    errors = {}
    completed = set()

    def update_response_message(panel_index, waiting=False):
        response_messages[panel_index].set_content(
            markdown=format_assistant_message(
                reasoning_outputs[panel_index], answers[panel_index], waiting=waiting
            )
        )

    with ThreadPoolExecutor(
        max_workers=PANEL_COUNT, thread_name_prefix="openrouter"
    ) as executor:
        futures = [
            executor.submit(
                stream_model,
                panel_index,
                model["id"],
                request_histories[panel_index],
                events,
                request_barrier,
            )
            for panel_index, model in enumerate(active_models)
        ]

        while len(completed) < PANEL_COUNT:
            event_type, panel_index, payload = events.get()
            if event_type == "started":
                update_response_message(panel_index, waiting=True)
            elif event_type == "reasoning":
                reasoning_outputs[panel_index] += payload
                update_response_message(panel_index)
            elif event_type == "token":
                answers[panel_index] += payload
                update_response_message(panel_index)
            elif event_type == "error":
                errors[panel_index] = payload
                answers[panel_index] = f"⚠️ **Request failed:** {payload}"
                update_response_message(panel_index)
            elif event_type == "done":
                completed.add(panel_index)

        for future in futures:
            future.result()

    for panel_index in range(PANEL_COUNT):
        answer = answers[panel_index]
        if not answer:
            answer = "⚠️ Request failed: The model returned an empty response."
        _arena_histories[panel_index].append(
            {
                "role": "assistant",
                "content": answer,
                "reasoning": reasoning_outputs[panel_index],
            }
        )